## 02_feature_engineering.ipynb: Time-Aware Feature Matrix Construction

This notebook focuses on building a highly engineered, time-aware feature matrix for our FIFA World Cup Predictor. We will generate features that capture team form, squad strength, and match-specific interactions, all while carefully avoiding lookahead bias.


### STEP 1: Dependencies & Data Initialization

We begin by importing essential libraries, mounting Google Drive to access our pre-processed data, and loading the historical match data and team index map. All date columns will be explicitly converted to datetime objects to ensure correct temporal operations.


In [5]:
# Import standard libraries for data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
from datetime import datetime

# Import tqdm for progress bars in loops
from tqdm.notebook import tqdm

# For mathematical operations like exponential decay
import math

# Set pandas display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Dependencies loaded.")

Dependencies loaded.


### STEP 2: Time-Weighted, Importance-Adjusted Rolling Form

This step involves creating a sophisticated function to calculate rolling form metrics for each team. These metrics are adjusted by match importance and a time decay factor, ensuring that recent and significant matches contribute more to a team's perceived form.

**Match Importance Multiplier:**
- World Cup Finals = 4.0
- Continental Tournaments (Euros, Copa América, etc.) = 3.0
- Qualifiers / Nations League = 2.0
- Friendlies = 1.0

**Exponential Time Decay:**
Weight = Importance * exp(-0.0005 * Days_Ago)

We will calculate:
- Weighted rolling win rate.
- Weighted rolling goals scored per match.
- Weighted rolling goals conceded per match.


### STEP 3: Squad Telemetry Aggregations

This section focuses on incorporating player-level data to create team-level squad metrics. We'll aggregate player statistics (e.g., FIFA OVR ratings, age, potential) to generate 'Squad_Core_Rating', 'Squad_Depth_Rating', 'Squad_Average_Age', and 'Squad_Experience_Index' for each national team. These metrics will then be mapped to our match dataframe as a static snapshot for the 2026 World Cup.


### STEP 4: Pairwise Match Interactions

To enhance our predictive models, we generate explicit interaction metrics that capture the relative strengths and weaknesses between the home and away teams for each match. These features provide a direct comparison, which is often very powerful for tree-based models.


### STEP 5: Final Sanity Check & Export

Before exporting, we perform a final sanity check on our engineered feature matrix. This includes verifying the dataframe's shape, ensuring no missing values exist in our new features, and visualizing the correlation of our top engineered features against the match outcome to confirm their potential predictive power. Finally, the processed dataframe will be saved for use in the next notebook.


In [6]:
# ==============================================================================
# PORTFOLIO-GRADE FIFA WORLD CUP 2026 PREDICTION SYSTEM
# WORKFLOW STAGE: NOTEBOOK 02 — FEATURE ENGINEERING & SYNTHESIS
# ==============================================================================

# Imports are now handled in the initial setup cell (2ac4c7e6)

print("🚀 Initializing Notebook 02 Feature Engineering Engine...")

# ==============================================================================
# STEP 1: MOUNT GOOGLE DRIVE & AUTOMATED INGESTION WITH ARCHIVE SCANNER
# ==============================================================================

# Try mounting drive if running inside Google Colab environment
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive mounted successfully.")
except ImportError:
    print("ℹ️ Running in local/non-Colab environment. Skipping Drive auto-mount.")

# Define standard search and destination directories
drive_base_dir = '/content/drive/MyDrive/FIFA_Predictor_Data/'
alternative_base_dir = '/content/drive/My Drive/FIFA_Predictor_Data/'
local_raw_dir = '/content/fifa-wc2026-predictor/data/raw/'

# Path resolution helper
base_dir = drive_base_dir if os.path.exists(drive_base_dir) else (alternative_base_dir if os.path.exists(alternative_base_dir) else './')
os.makedirs(local_raw_dir, exist_ok=True)

match_data_path = os.path.join(base_dir, 'df_matches_filtered.csv')
player_data_dest = os.path.join(base_dir, 'player_data/players_2020.csv')

# --- Defensive Archive Extraction Utility for Missing Files ---
if not os.path.exists(player_data_dest):
    print("🔍 'players_2020.csv' not found at target path. Scanning Drive for compressed archives...")
    # Search common upload paths for compressed player archives
    search_paths = [base_dir, '/content/drive/MyDrive/', '/content/drive/My Drive/', './']
    extracted = False

    for s_path in search_paths:
        if os.path.exists(s_path):
            for root, dirs, files in os.walk(s_path):
                for file in files:
                    if file.endswith('.zip') and ('player' in file.lower() or 'stats' in file.lower() or 'fifa' in file.lower()):
                        zip_full_path = os.path.join(root, file)
                        try:
                            with zipfile.ZipFile(zip_full_path, 'r') as z:
                                target_files = [f for f in z.namelist() if 'player' in f.lower() and f.endswith('.csv')]
                                if target_files:
                                    print(f"📦 Found player archive target inside: {zip_full_path}")
                                    os.makedirs(os.path.dirname(player_data_dest), exist_ok=True)
                                    # Extract file
                                    with open(player_data_dest, 'wb') as f_out:
                                        f_out.write(z.read(target_files[0]))
                                    print(f"✅ Successfully extracted & aligned to: {player_data_dest}")
                                    extracted = True
                                    break
                        except Exception as e:
                            continue
                if extracted: break
        if extracted: break

# Load Cleaned Historical Matches Base Matrix
if not os.path.exists(match_data_path):
    # Search safety mechanism
    print(f"⚠️ Primary match path missing. Scanning current directory...")
    if os.path.exists('df_matches_filtered.csv'):
        match_data_path = 'df_matches_filtered.csv'
    else:
        raise FileNotFoundError(f"❌ Critical Error: 'df_matches_filtered.csv' could not be found. Please check Notebook 01 output.")

df_matches = pd.read_csv(match_data_path)
df_matches['date'] = pd.to_datetime(df_matches['date'])
print(f"✅ Historical match base active. Shape: {df_matches.shape}")

# Load and process player statistics if available
squad_features_dict = {}
if os.path.exists(player_data_dest):
    try:
        print("📊 Ingesting actual player data telemetry...")
        df_players = pd.read_csv(player_data_dest)
        nat_col = 'nationality' if 'nationality' in df_players.columns else ('nationality_name' if 'nationality_name' in df_players.columns else None)

        if nat_col:
            # Process static baseline team telemetry matrices from player pools
            df_squad_agg = df_players.groupby(nat_col).agg(
                squad_overall_mean=('overall', 'mean'),
                squad_overall_max=('overall', 'max'),
                squad_age_mean=('age', 'mean')
            ).reset_index()
            # Map into dictionary index for lookups
            for _, row in df_squad_agg.iterrows():
                squad_features_dict[str(row[nat_col]).strip()] = {
                    'squad_overall_mean': row['squad_overall_mean'],
                    'squad_overall_max': row['squad_overall_max'],
                    'squad_age_mean': row['squad_age_mean']
                }
            print("✅ Player data successfully processed into core squad strength features.")
        else:
            print("⚠️ Player dataset structural anomaly: Nationality metadata columns missing.")
    except Exception as e:
        print(f"⚠️ Error parsing player csv: {e}. Activating fallback data engineering structures.")
else:
    print("ℹ️ Player data csv absent. Relying on rolling performance metrics for proxy strength.")

# ==============================================================================
# STEP 2 & 3: TIME-DECAYED ROLLING FORM & CHRONOLOGICAL SQUAD TELEMETRY
# ==============================================================================
print("⏳ Computing chronological time-decayed performance profiles (Strictly Lookahead-Safe)...")

# Transform matches into long-form logs to calculate historic performance vectors
home_logs = df_matches[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral']].copy()
home_logs.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament', 'neutral']
home_logs['is_home'] = 1

away_logs = df_matches[['date', 'away_team', 'home_team', 'away_score', 'home_score', 'tournament', 'neutral']].copy()
away_logs.columns = ['date', 'team', 'opponent', 'goals_for', 'goals_against', 'tournament', 'neutral']
away_logs['is_home'] = 0

df_long_logs = pd.concat([home_logs, away_logs], axis=0).sort_values(by=['team', 'date']).reset_index(drop=True)

# Calculate outcome points
df_long_logs['points'] = np.where(df_long_logs['goals_for'] > df_long_logs['goals_against'], 3,
                           np.where(df_long_logs['goals_for'] == df_long_logs['goals_against'], 1, 0))
df_long_logs['clean_sheet'] = np.where(df_long_logs['goals_against'] == 0, 1, 0)

# Define tournament weights dict based on match importance context
def compute_match_importance(tourney):
    t_lower = str(tourney).lower()
    if 'fifa world cup' in t_lower and 'qualifying' not in t_lower:
        return 4.0 # Elite Tier
    elif any(x in t_lower for x in ['copa américa', 'copa america', 'euro', 'nations league', 'qualifying', 'championship']):
        return 3.0 # Highly Competitive
    elif 'friendly' in t_lower:
        return 1.0 # Low Weight Exhibition
    else:
        return 2.0 # Standard Continental/Sub-tournaments

df_long_logs['importance_weight'] = df_long_logs['tournament'].apply(compute_match_importance)

# Custom chronological lookback generator (Window sizing: 8 matches)
pre_match_features_list = []
lambda_decay = 0.0015  # Exponential lambda decay (scales smoothly over days)

for team, group in df_long_logs.groupby('team'):
    group = group.sort_values('date').reset_index(drop=True)
    for idx in range(len(group)):
        target_date = group.loc[idx, 'date']

        # Enforce chronological isolation: Extract records strictly BEFORE the target match date
        historical_pool = group[group['date'] < target_date].tail(8)

        if historical_pool.empty:
            # Baseline neutral default variables for early structural history
            rolling_form = 1.0
            rolling_goals_for = 0.5
            rolling_goals_against = 1.5
            rolling_clean_sheet_rate = 0.1
        else:
            # Calculate time delta in days between past matches and the current match row
            day_deltas = (target_date - historical_pool['date']).dt.days

            # Formulate composite mathematical weights: Tournament Importance x Time Decay
            composite_weights = historical_pool['importance_weight'] * np.exp(-lambda_decay * day_deltas)
            weight_denominator = composite_weights.sum()

            if weight_denominator > 0:
                rolling_form = (historical_pool['points'] * composite_weights).sum() / weight_denominator
                rolling_goals_for = (historical_pool['goals_for'] * composite_weights).sum() / weight_denominator
                rolling_goals_against = (historical_pool['goals_against'] * composite_weights).sum() / weight_denominator
                rolling_clean_sheet_rate = (historical_pool['clean_sheet'] * composite_weights).sum() / weight_denominator
            else:
                rolling_form = historical_pool['points'].mean()
                rolling_goals_for = historical_pool['goals_for'].mean()
                rolling_goals_against = historical_pool['goals_against'].mean()
                rolling_clean_sheet_rate = historical_pool['clean_sheet'].mean()

        pre_match_features_list.append({
            'date': target_date,
            'team': team,
            'rolling_form': rolling_form,
            'rolling_goals_for': rolling_goals_for,
            'rolling_goals_against': rolling_goals_against,
            'rolling_clean_sheets': rolling_clean_sheet_rate
        })

df_engineered_history = pd.DataFrame(pre_match_features_list)

# ==============================================================================
# STEP 4: MERGE FIELDS & CAPTURE PAIRWISE INTERACTIONS
# ==============================================================================
print("⚔️ Fusing records and computing tactical clashes and differential interfaces...")

# Map features back into wide format for Home teams
df_master = df_matches.merge(
    df_engineered_history.rename(columns=lambda x: f'home_{x}' if x not in ['date', 'team'] else x),
    left_on=['date', 'home_team'], right_on=['date', 'team'], how='left'
).drop(columns=['team'])

# Map features back into wide format for Away teams
df_master = df_master.merge(
    df_engineered_history.rename(columns=lambda x: f'away_{x}' if x not in ['date', 'team'] else x),
    left_on=['date', 'away_team'], right_on=['date', 'team'], how='left'
).drop(columns=['team'])

# Synthesize static player telemetry properties safely mapped by index
def assign_squad_metric(team_name, metric_key, fallback_val):
    clean_name = str(team_name).strip()
    if clean_name in squad_features_dict:
        return squad_features_dict[clean_name][metric_key]
    return fallback_val

# Extract player-level strength metrics or impute standard uniform defaults if file was absent
df_master['home_squad_overall_mean'] = df_master['home_team'].apply(lambda x: assign_squad_metric(x, 'squad_overall_mean', 72.0))
df_master['away_squad_overall_mean'] = df_master['away_team'].apply(lambda x: assign_squad_metric(x, 'squad_overall_mean', 72.0))
df_master['home_squad_overall_max'] = df_master['home_team'].apply(lambda x: assign_squad_metric(x, 'squad_overall_max', 85.0))
df_master['away_squad_overall_max'] = df_master['away_team'].apply(lambda x: assign_squad_metric(x, 'squad_overall_max', 85.0))
df_master['home_squad_age_mean'] = df_master['home_team'].apply(lambda x: assign_squad_metric(x, 'squad_age_mean', 26.5))
df_master['away_squad_age_mean'] = df_master['away_team'].apply(lambda x: assign_squad_metric(x, 'squad_age_mean', 26.5))

# Generate Interaction Metrics
df_master['form_difference'] = df_master['home_rolling_form'] - df_master['away_rolling_form']
df_master['attack_vs_defense_clash'] = df_master['home_rolling_goals_for'] - df_master['away_rolling_goals_against']
df_master['away_attack_vs_home_defense_clash'] = df_master['away_rolling_goals_for'] - df_master['home_rolling_goals_against']
df_master['squad_quality_difference'] = df_master['home_squad_overall_mean'] - df_master['away_squad_overall_mean']

# Map the explicit Neutral Venue Flag safely
if 'neutral' in df_master.columns:
    df_master['neutral_venue_flag'] = df_master['neutral'].astype(int)
else:
    df_master['neutral_venue_flag'] = 0

# ==============================================================================
# STEP 5: CLEAN COMPILATION, RE-ENFORCE WALL & MASTER FILE EXPORT
# ==============================================================================

# Impute any localized NaN artifacts using safe historical medians
numeric_cols = df_master.select_dtypes(include=[np.number]).columns
df_master[numeric_cols] = df_master[numeric_cols].fillna(df_master[numeric_cols].median())

# Enforce the strict May 31, 2026 chronological wall across all records
temporal_cutoff = pd.to_datetime('2026-05-31')
df_master = df_master[df_master['date'] <= temporal_cutoff].reset_index(drop=True)

# Export finalized dataset matrix
output_export_path = os.path.join(base_dir, 'final_engineered_feature_matrix.csv')
os.makedirs(os.path.dirname(output_export_path), exist_ok=True)
df_master.to_csv(output_export_path, index=False)

print("\n" + "="*80)
print("🎯 NOTEBOOK 02 RUN COMPLETION STATUS: SUCCESS")
print("="*80)
print(f"📁 Master Feature Matrix exported to: {output_export_path}")
print(f"📊 Final Dataframe Shape: {df_master.shape}")
print("-"*80)
print("📝 Preview of newly engineered high-signal predictive columns:")
preview_cols = ['date', 'home_team', 'away_team', 'form_difference',
                'attack_vs_defense_clash', 'squad_quality_difference', 'neutral_venue_flag']
print(df_master[preview_cols].head())
print("="*80)

🚀 Initializing Notebook 02 Feature Engineering Engine...
Mounted at /content/drive
✅ Google Drive mounted successfully.
✅ Historical match base active. Shape: (49285, 9)
📊 Ingesting actual player data telemetry...
✅ Player data successfully processed into core squad strength features.
⏳ Computing chronological time-decayed performance profiles (Strictly Lookahead-Safe)...
⚔️ Fusing records and computing tactical clashes and differential interfaces...

🎯 NOTEBOOK 02 RUN COMPLETION STATUS: SUCCESS
📁 Master Feature Matrix exported to: /content/drive/MyDrive/FIFA_Predictor_Data/final_engineered_feature_matrix.csv
📊 Final Dataframe Shape: (49619, 28)
--------------------------------------------------------------------------------
📝 Preview of newly engineered high-signal predictive columns:
        date home_team away_team  form_difference  attack_vs_defense_clash  \
0 1872-11-30  Scotland   England         0.000000                     -1.0   
1 1873-03-08   England  Scotland         0.0000

In [7]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# ==============================================================================
# STEP 1: PROGRAMMATIC FOLDER VIRTUALIZATION & REORGANIZATION
# ==============================================================================

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Define paths
base_path = '/content/drive/MyDrive/fifa-wc2026-predictor/data_pipeline/02_feature_engineering/'
subfolders = ['input', 'output', 'visuals']

for folder in subfolders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

# Locate and migrate master dataset
# Source was previously defined in the notebook as output_export_path
current_dataset_path = '/content/drive/MyDrive/FIFA_Predictor_Data/final_engineered_feature_matrix.csv'
target_dataset_path = os.path.join(base_path, 'output', 'final_engineered_feature_matrix.csv')

if os.path.exists(current_dataset_path):
    shutil.copy2(current_dataset_path, target_dataset_path)
    print(f"✅ Dataset migrated to: {target_dataset_path}")

print(f"📂 Storage Architecture at: {base_path}")
for folder in subfolders:
    print(f"  └── {folder}/")

# ==============================================================================
# STEP 2: VISUAL 4 — CUSTOM TIME-DECAY WEIGHTING FUNCTION CURVE
# ==============================================================================

def generate_time_decay_plot():
    lambda_decay = 0.0015
    days = np.linspace(0, 3650, 500) # 10 years
    weights = np.exp(-lambda_decay * days)

    plt.figure(figsize=(10, 6), dpi=300)
    sns.set_style("whitegrid")

    plt.plot(days / 365, weights, color='#2c3e50', linewidth=2.5, label='Decay Curve')
    plt.fill_between(days / 365, weights, color='#34495e', alpha=0.15)

    plt.title('Temporal Decay of Historical Match Relevance', fontsize=14, fontweight='bold')
    plt.xlabel('Years Elapsed Since Match', fontsize=12)
    plt.ylabel('Weight Multiplier (Signal Retention)', fontsize=12)

    plt.annotate('Maximum Weight (Recent Form)', xy=(0, 1), xytext=(1.5, 0.9),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5))

    plt.tight_layout()
    plt.savefig(os.path.join(base_path, 'visuals/04_time_decay_function.png'))
    plt.close()

generate_time_decay_plot()

# ==============================================================================
# STEP 3: VISUAL 5 — DELTA FEATURE INTERACTION HEATMAP
# ==============================================================================

def generate_interaction_heatmap(data_path):
    df = pd.read_csv(data_path)

    # Isolate interaction and rolling features
    interaction_cols = [
        'form_difference', 'attack_vs_defense_clash',
        'away_attack_vs_home_defense_clash', 'squad_quality_difference',
        'home_rolling_form', 'away_rolling_form',
        'home_rolling_goals_for', 'away_rolling_goals_for'
    ]

    # Ensure columns exist before correlating
    valid_cols = [c for c in interaction_cols if c in df.columns]
    corr_matrix = df[valid_cols].corr()

    plt.figure(figsize=(12, 10), dpi=300)
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='vlag', center=0,
                linewidths=.5, cbar_kws={"shrink": .8})

    plt.title('Correlation Matrix of Engineered Interaction Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(base_path, 'visuals/05_feature_interaction_heatmap.png'))
    plt.close()

generate_interaction_heatmap(target_dataset_path)

# ==============================================================================
# STEP 4: VISUAL 6 — COMPETITIVE CONTEXTUAL WEIGHTING
# ==============================================================================

def generate_tier_weighting_plot():
    tiers = {
        'World Cup Finals': 4.0,
        'Continental Cups / Qualifiers': 3.0,
        'Nations League / Sub-Tournaments': 2.0,
        'International Friendlies': 1.0
    }

    tier_df = pd.DataFrame(list(tiers.items()), columns=['Tournament Tier', 'Importance Multiplier'])

    plt.figure(figsize=(10, 6), dpi=300)
    ax = sns.barplot(x='Importance Multiplier', y='Tournament Tier', data=tier_df, palette='Blues_d')

    plt.title('Match Importance Weighting per Tournament Tier', fontsize=14, fontweight='bold')
    plt.xlabel('Weight Multiplier', fontsize=12)
    plt.ylabel('Tournament Category', fontsize=12)

    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(base_path, 'visuals/06_tier_weighting_distribution.png'))
    plt.close()

generate_tier_weighting_plot()

print("📊 Presentation graphics generated and saved to 'visuals/' directory.")

Mounted at /content/drive
✅ Dataset migrated to: /content/drive/MyDrive/fifa-wc2026-predictor/data_pipeline/02_feature_engineering/output/final_engineered_feature_matrix.csv
📂 Storage Architecture at: /content/drive/MyDrive/fifa-wc2026-predictor/data_pipeline/02_feature_engineering/
  └── input/
  └── output/
  └── visuals/


/tmp/ipykernel_480/311363050.py:109: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x='Importance Multiplier', y='Tournament Tier', data=tier_df, palette='Blues_d')


📊 Presentation graphics generated and saved to 'visuals/' directory.
